#### Problem statement

Predict the political party from the tweet text and the handle

#### Data description
This dataset has three columns - label (party name), twitter handle, tweet text


#### Problem Description:

Design a feed forward deep neural network to predict the political party using the pytorch or tensorflow. 
Build two models

1. Without using the handle

2. Using the handle


#### Deliverables

- Report the performance on the test set.

- Try multiple models and with different hyperparameters. Present the results of each model on the test set. No need to create a dev set.

- Experiment with:
    -L2 and dropout regularization techniques
    -SGD, RMSProp and Adam optimization techniques



- Creating a fixed-sized vocabulary: Give a unique id to each word in your selected vocabulary and use it as the input to the network

    - Option 1: Feed-forward networks can only handle fixed-sized inputs. You can choose to have a fixed-sized K words from the tweet text (e.g. the first K word, randomly selected K word etc.). K can be a hyperparameter. 

    - Option 2: you can choose top N (e.g. N=1000) frequent words from the dataset and use an N-sized input layer. If a word is present in a tweet, pass the id, 0 otherwise
    
    -  Clearly state your design choices and assumptions. Think about the pros and cons of each option.

 

<b> Tabulate your results, either at the end of the code file or in the text box on the submission page. The final result should have:</b>

1. Experiment description

2. Hyperparameter used and their values

3. Performance on the test set

 

## Selected Option 2: Choosing top N frequent words
### Design Choice & Pros - Cons

I have selected Top N Frequent Words as Binary Features using Tf-IDF because of the following Pros:
1. Easy to Implement
2. Frequent Keywords are High-Impact for sentiment analysis
3. Flexibility to select the number of words e.g N=1000, 500 as a parameter to tune the model

Cons:
1. Ignores the sequence with which words occur i.e. the order of the words.
2. Rare words which occur less infrequently but are important may be ignored.
3. Stopword Noise - Top-N words may include uninformative terms without pre-processing

### Assumptions

Tweets may include emojis, URLs, mentions which may need pre-processing.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7" 

In [2]:
import pandas as pd
import torch

/tmp/ipykernel_2147395/1316919942.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
torch.cuda.is_available()

True

In [4]:
print(torch.cuda.device_count())

8


In [5]:
device = torch.device('cuda:0')

In [6]:
from sklearn.model_selection import train_test_split

train_df = pd.read_csv('train.csv')
train_df.columns
test_df = pd.read_csv('test.csv')
test_df.columns

test_df = test_df.dropna()
train_df = train_df.dropna()


print("Training set shape:", train_df.shape)
#print("Validation set shape:", valid_df.shape)

Training set shape: (72734, 4)


In [7]:
import re
import emoji
from wordsegment import load, segment

load()  # Load wordsegment for hashtag splitting

def preprocess_tweet(tweet):
    # Remove URLs and mentions
    tweet = re.sub(r'http\S+', 'URL', tweet)
    tweet = re.sub(r'@\w+', 'MENTION', tweet)
    
    # Split hashtags into words (e.g., #Democrat2020 → democrat 2020)
    tweet = re.sub(r'#(\w+)', lambda x: ' '.join(segment(x.group(1))), tweet)
    
    # Convert emojis to text (😠 → ":angry_face:")
    tweet = emoji.demojize(tweet)
    
    # Lowercase (optional, but done here to avoid redundant steps)
    tweet = tweet.lower()
    
    return tweet

# Preprocess all data upfront
train_df['Tweet'] = train_df['Tweet'].apply(preprocess_tweet)
test_df['Tweet'] = test_df['Tweet'].apply(preprocess_tweet)

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
import torch

N = 5000

vectorizer = CountVectorizer(binary=True, max_features=N, stop_words='english', strip_accents='unicode',lowercase=True)

X_train = vectorizer.fit_transform(train_df['Tweet'].fillna(''))
X_test = vectorizer.transform(test_df['Tweet'].fillna(''))
#X_Val = vector.fit_transform(valid_df['Tweet'].fillna(''))

# Convert to tensor
X_train_tensor = torch.FloatTensor(X_train.toarray())
X_test_tensor = torch.FloatTensor(X_test.toarray())
#X_Val = torch.FloatTensor(X_Val.toarray())

label_map = {'Democrat': 0, 'Republican': 1}
y_train = torch.LongTensor([label_map[label] for label in train_df['Party']])
y_test = torch.LongTensor([label_map[label] for label in test_df['Party']])
#y_val = torch.LongTensor([label_map[label] for label in valid_df['Party']])

X_train_tensor = X_train_tensor.to(device)
y_train = y_train.to(device)

X_test_tensor = X_test_tensor.to(device)
y_test = y_test.to(device)


In [9]:
import torch.nn as nn
import torch.optim as optim

class FFN(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 128,64], dropout=0.2):
        super(FFN, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], hidden_sizes[2]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[2], hidden_sizes[3]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_sizes[3], 2),
        )

    def forward(self, x):
        return self.layers(x)

In [10]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test, optimizer_type, dropout, l2, learning_rate=0.001, epochs=10, batch_size=32, eval_interval=1):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    if optimizer_type == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=l2)
    elif optimizer_type == "RMSProp":
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=l2)
    else:
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2)
    
    X_train, y_train, X_test, y_test = X_train.to(device), y_train.to(device), X_test.to(device), y_test.to(device)
    
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    def evaluate(X, y):
        model.eval()
        with torch.no_grad():
            outputs = model(X)
            _, predicted = torch.max(outputs, 1)
            return (predicted == y).sum().item() / y.size(0)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch + 1) % eval_interval == 0:
            train_acc = evaluate(X_train, y_train)
            test_acc = evaluate(X_test, y_test)
            
    return train_acc, test_acc


In [11]:
configurations = [
    {'optimizer': 'SGD', 'dropout': 0.0, 'l2': 0.0},
    {'optimizer': 'SGD', 'dropout': 0.3, 'l2': 1e-4},
    {'optimizer': 'RMSProp', 'dropout': 0.0, 'l2': 0.0},
    {'optimizer': 'RMSProp', 'dropout': 0.3, 'l2': 1e-4},
    {'optimizer': 'Adam', 'dropout': 0.0, 'l2': 0.0},
    {'optimizer': 'Adam', 'dropout': 0.3, 'l2': 1e-4},
]

In [ ]:
results = []
hidden_sizes=[512, 256, 256, 128]
epochs = 6
learning_rate = 0.00003
batch_size = 16
    
for config in configurations:
    model = FFN(X_train.shape[1],hidden_sizes,dropout=config['dropout']).to(device)
    train_acc, test_acc = train_and_evaluate(model, X_train_tensor, y_train, X_test_tensor, y_test, optimizer_type=config['optimizer'], dropout=config['dropout'], l2=config['l2'], learning_rate=learning_rate, epochs=epochs, batch_size=batch_size,eval_interval = 1)
    
    results.append({
        'optimizer': config['optimizer'],
        'dropout': config['dropout'],
        'l2': config['l2'],
        'test_accuracy': test_acc,
        'train_accuracy': train_acc
    })

df = pd.DataFrame(results)

In [33]:
df

,optimizer,dropout,l2,test_accuracy,train_accuracy
0,SGD,0.0,0.0000,0.506047,0.514835
1,SGD,0.3,0.0001,0.506047,0.514835
2,RMSProp,0.0,0.0000,0.739473,0.900225
3,RMSProp,0.3,0.0001,0.735830,0.818861
4,Adam,0.0,0.0000,0.742678,0.988066
5,Adam,0.3,0.0001,0.748215,0.964336


## Model 2: Using Handle

We use label encoder to encode the handles and train the model

In [19]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_df['Handle_encoded'] = label_encoder.fit_transform(train_df['Handle'])
test_df['Handle_encoded'] = label_encoder.transform(test_df['Handle'])

X_train_handle_tensor = torch.FloatTensor(train_df['Handle_encoded'].values).unsqueeze(1)
X_test_handle_tensor = torch.FloatTensor(test_df['Handle_encoded'].values).unsqueeze(1)

X_train_handle_tensor = X_train_handle_tensor.to(device)
X_test_handle_tensor = X_test_handle_tensor.to(device)

X_train_combined = torch.cat((X_train_tensor, X_train_handle_tensor), dim=1)
X_test_combined = torch.cat((X_test_tensor, X_test_handle_tensor), dim=1)


X_train_combined = X_train_combined.to(device)
X_test_combined = X_test_combined.to(device)


In [20]:
input_size_combined = X_train_combined.shape[1]

In [21]:
configurations = [
    {'optimizer': 'SGD', 'dropout': 0.0, 'l2': 0.0},
    {'optimizer': 'SGD', 'dropout': 0.3, 'l2': 1e-4},
    {'optimizer': 'RMSProp', 'dropout': 0.0, 'l2': 0.0},
    {'optimizer': 'RMSProp', 'dropout': 0.3, 'l2': 1e-4},
    {'optimizer': 'Adam', 'dropout': 0.0, 'l2': 0.0},
    {'optimizer': 'Adam', 'dropout': 0.3, 'l2': 1e-4},
]

In [ ]:
results_with_handle = []
hidden_sizes=[512, 256, 256, 128]
epochs = 10
learning_rate = 0.00003
batch_size = 16
    
for config in configurations:
    model = FFN(input_size_combined ,hidden_sizes,dropout=config['dropout']).to(device)
    train_acc, test_acc = train_and_evaluate(model, X_train_combined, y_train, X_test_combined, y_test, optimizer_type=config['optimizer'], dropout=config['dropout'], l2=config['l2'], learning_rate=learning_rate, epochs=epochs, batch_size=batch_size,eval_interval = 1)
    
    results_with_handle.append({
        'optimizer': config['optimizer'],
        'dropout': config['dropout'],
        'l2': config['l2'],
        'test_accuracy': test_acc,
        'train_accuracy': train_acc
    })


In [28]:
df_with_handle = pd.DataFrame(results_with_handle)

In [32]:
df_with_handle

,optimizer,dropout,l2,test_accuracy,train_accuracy
0,SGD,0.0,0.0000,0.514789,0.524157
1,SGD,0.3,0.0001,0.502768,0.491256
2,RMSProp,0.0,0.0000,0.747487,0.789301
3,RMSProp,0.3,0.0001,0.731386,0.756373
4,Adam,0.0,0.0000,0.729273,0.793467
5,Adam,0.3,0.0001,0.751566,0.800245
